# Longitudinal Panel Econometrics & Policy Estimation Engine
### Pooled OLS | Fixed Effects (linearmodels.panel.PanelOLS & CRVE) | Random Effects (Swamy-Arora FGLS) | Hausman Specification Test

This econometric modeling and statistical inference pipeline demonstrates:
1. **Balanced Longitudinal Panel Structure:** Evaluating **120 countries observed across 10 years (1,200 country-year observations)**.
2. **Industry-Standard Specification Tournament:** Using `linearmodels` to compare Pooled OLS (`PooledOLS`), Fixed Effects (`PanelOLS` with `EntityEffects` and Liang-Zeger Cluster-Robust Standard Errors / CRVE), and Random Effects (`RandomEffects` with quasi-demeaning parameter $\theta = 0.8896$).
3. **Spectrally-Decomposed Hausman Test:** Executing classical asymptotic specification testing ($\chi^2 = 24.63, p < 0.001$), formally rejecting $H_0$ to confirm that unobserved country heterogeneity introduces endogeneity, establishing **Fixed Effects as the consistent estimator**.

In [1]:
import os
import sys
import numpy as np
import pandas as pd

# Add root directory to path
sys.path.insert(0, os.getcwd())

from src.data_loader import PanelDataLoader
from src.panel_econometric_engine import PanelEconometricEngine

# 1. Ingest Balanced Macroeconomic Panel Dataset
loader = PanelDataLoader(data_dir="data", n_countries=120, n_years=10, random_state=42)
df = loader.load_panel_data()

engine = PanelEconometricEngine(df, entity_col='country_id', time_col='year', dep_var='log_exports')
print(f"Total Panel Observations : {len(df):,}")
print(f"Distinct Country Entities: {df['country_id'].nunique()}")
print(f"Longitudinal Time Periods: {df['year'].nunique()} years ({df['year'].min()} - {df['year'].max()})")

Total Panel Observations : 1,200
Distinct Country Entities: 120
Longitudinal Time Periods: 10 years (2014 - 2023)


## 2. Industry-Standard Econometric Estimations (linearmodels PooledOLS, PanelOLS, RandomEffects)

In [3]:
pols = engine.estimate_pooled_ols()
fe = engine.estimate_fixed_effects()
re = engine.estimate_random_effects()

vars_list = ['TFI Score', 'Log GDP', 'Tariff Rate', 'Infra Score', 'FX Volatility']
rows = []
for i, v in enumerate(vars_list):
    rows.append({
        'Variable': v,
        'Pooled OLS Coef (SE)': f"{pols['coefficients'][i]:.4f} ({pols['std_errors'][i]:.4f})",
        'Fixed Effects Coef (CRVE)': f"{fe['coefficients'][i]:.4f} ({fe['std_errors'][i]:.4f})",
        'Swamy-Arora RE Coef (SE)': f"{re['coefficients'][i]:.4f} ({re['std_errors'][i]:.4f})"
    })

res_table = pd.DataFrame(rows)
print("=" * 95)
print("LONGITUDINAL PANEL ECONOMETRIC REGRESSION ESTIMATES (LINEARMODELS)")
print("=" * 95)
print(res_table.to_string(index=False))
print("-" * 95)
print(f"• Fixed Effects sigma_e^2   : {fe['sigma_e2']:.4f} | Within R^2: {fe['r_squared']:.4f}")
print(f"• Swamy-Arora sigma_u^2     : {re['sigma_u2']:.4f} | Quasi-Demeaning theta: {re['theta']:.4f}")
print("=" * 95)

LONGITUDINAL PANEL ECONOMETRIC REGRESSION ESTIMATES (LINEARMODELS)
     Variable Pooled OLS Coef (SE) Fixed Effects Coef (CRVE) Swamy-Arora RE Coef (SE)
    TFI Score      1.5082 (0.3788)           1.2584 (0.1671)          1.1800 (0.1343)
      Log GDP      0.8687 (0.0244)           0.5704 (0.1626)          0.8219 (0.0677)
  Tariff Rate     -0.0277 (0.0163)          -0.0342 (0.0057)         -0.0311 (0.0057)
  Infra Score      0.1609 (0.1623)           0.2139 (0.0630)          0.1842 (0.0569)
FX Volatility     -0.3509 (0.4844)          -1.0818 (0.1772)         -1.0778 (0.1710)
-----------------------------------------------------------------------------------------------
• Fixed Effects sigma_e^2   : 0.1603 | Within R^2: 0.2802
• Swamy-Arora sigma_u^2     : 1.2994 | Quasi-Demeaning theta: 0.8896


## 3. Spectrally-Decomposed Hausman Specification Test

In [5]:
hausman = engine.run_hausman_specification_test()
print("=" * 95)
print("HAUSMAN SPECIFICATION TEST REPORT")
print("=" * 95)
print(f"• Hausman Chi-Square Statistic : {hausman['hausman_statistic']:.2f}")
print(f"• Degrees of Freedom           : {hausman['degrees_of_freedom']}")
print(f"• Asymptotic p-value           : {hausman['p_value']} (p < 0.0001)")
print(f"• Hypothesis Verdict           : {hausman['hypothesis_verdict']}")
print(f"• Preferred Econometric Model  : {hausman['preferred_model']}")
print("=" * 95)

HAUSMAN SPECIFICATION TEST REPORT
• Hausman Chi-Square Statistic : 24.63
• Degrees of Freedom           : 2
• Asymptotic p-value           : 4e-06 (p < 0.0001)
• Hypothesis Verdict           : Reject H0 (Presence of Endogeneity)
• Preferred Econometric Model  : Fixed Effects (Within)
